# Options Desk Analytics (ODX) - Case Study

In this case study we will:
1. Load an option chain using the Polygon data adapter.
2. Clean the chain data using ODX cleaning pipeline.
3. Fit an SSVI surface to the cleaned chain.
4. Price a Barrier Option using local volatility concepts or Black-Scholes.
5. Run a delta-hedging backtest simulation.

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from odx.data.polygon import PolygonDataSource
from odx.data.cleaning import clean_chain
from odx.vol.ssvi import fit_ssvi_surface, ssvi_total_variance
from odx.pricers.analytic.bs import bs_price

plt.style.use("ggplot")

## 1. Load Data
We use a sandbox/mock adapter for demonstration purposes.

In [ ]:
adapter = PolygonDataSource(api_key="DEMO_KEY", sandbox=True)
# In a real environment, you would call get_option_chain("SPY")
# Here we mock a tiny synthetic chain
spot = 100.0
chain_data = []
for T in [0.25, 0.5, 1.0]:
    for K in [90.0, 95.0, 100.0, 105.0, 110.0]:
        vol = 0.2 + 0.05 * (K / spot - 1)**2
        price = bs_price(spot, K, T, 0.05, vol, "call", 0.0)
        chain_data.append({
            "underlying": "SYNTH",
            "expiry": (datetime.date.today() + datetime.timedelta(days=int(T*365))).strftime("%Y-%m-%d"),
            "cp": "call",
            "K": K,
            "T": T,
            "spot": spot,
            "r": 0.05,
            "q": 0.0,
            "bid": max(0.01, price - 0.05),
            "ask": price + 0.05,
            "mid": price,
            "volume": 100,
            "openInterest": 500
        })
df_chain = pd.DataFrame(chain_data)
df_chain.head()

## 2. Clean Data
Clean the chain to ensure no crossed markets, negative T, or zero volume/OI options impact our calibration.

In [ ]:
df_clean, df_flagged = clean_chain(df_chain, drop_zero_oi=True, drop_zero_volume=True)
print(f"Cleaned: {len(df_clean)} rows, Flagged: {len(df_flagged)} rows.")

## 3. Fit SSVI Surface
Calibrate Gatheral's SSVI model to our cleaned implied volatility data.

In [ ]:
# ODX fit_ssvi_surface expects "F" and "iv" columns.
df_clean["F"] = df_clean["spot"] * np.exp((df_clean["r"] - df_clean["q"]) * df_clean["T"])
# Synthetic IV roughly recovered
df_clean["iv"] = 0.2 + 0.05 * (df_clean["K"] / df_clean["spot"] - 1)**2

params, rmse, info = fit_ssvi_surface(df_clean, check_arb=True)
print("Fitted Parameters (A, B, rho, eta, gamma):", params)
print("Arbitrage Free:", info.get("arbitrage_free"))

## 4. Barrier Option Pricing
Pricing a Down-and-Out Call under Black-Scholes using our extracted ATM Volatility.

In [ ]:
# Down-and-Out Call formula
S, K, H, T, r, q, sigma = 100.0, 100.0, 95.0, 1.0, 0.05, 0.0, 0.20

lam = (r - q + 0.5 * sigma**2) / sigma**2
y = np.log(H**2 / (S * K)) / (sigma * np.sqrt(T)) + lam * sigma * np.sqrt(T)

from scipy.stats import norm
d1 = (np.log(S/K) + (r - q + 0.5*sigma**2)*T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)

x1 = (np.log(S/H) + (r - q + 0.5*sigma**2)*T) / (sigma * np.sqrt(T))
y1 = (np.log(H/S) + (r - q + 0.5*sigma**2)*T) / (sigma * np.sqrt(T))

c = S * np.exp(-q*T) * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)
c_di = S * np.exp(-q*T) * (H/S)**(2*lam) * norm.cdf(y) - K * np.exp(-r*T) * (H/S)**(2*lam-2) * norm.cdf(y - sigma * np.sqrt(T))

price_doc = c - c_di
print(f"Down-and-Out Call Price: {price_doc:.4f}")

## 5. Delta Hedging Backtest
A basic event-driven simulation loop.

In [ ]:
# Assume daily rebalancing over 10 days
spots = np.linspace(100, 105, 10)
dt = 1/252
portfolio_val = []
cash = 0.0
shares = 0.0

for i, st in enumerate(spots):
    time_to_expiry = 1.0 - i * dt
    opt_price = bs_price(st, K, time_to_expiry, r, sigma, "call", q)
    
    from odx.greeks.analytic import bs_delta
    delta = bs_delta(st, K, time_to_expiry, r, sigma, "call", q)
    
    # Sell 1 option on day 1
    if i == 0:
        cash += opt_price
        
    # Delta hedge: buy shares to be delta neutral
    shares_needed = delta - shares
    cash -= shares_needed * st
    shares += shares_needed
    
    # Total portfolio val (Short 1 option + shares + cash)
    val = -opt_price + shares * st + cash
    portfolio_val.append(val)

plt.plot(portfolio_val)
plt.title("Delta-Hedged Short Call PnL")
plt.xlabel("Days")
plt.ylabel("PnL")
plt.show()